# Views: triangles, bands and permutations

A factorization stores L and U packed in one matrix, a symmetric solver
reads only one triangle, and pivoting reorders rows. MTL5's views express each
of these without writing a new algorithm: `upper`, `lower` and their strict
forms; `banded`; `hermitian`; and `map`, which selects and reorders rows and
columns.

**Uses:** `mtl5.view` (`upper`, `lower`, `strict_upper`, `strict_lower`,
`banded`, `hermitian`, `map`), `mtl5.trsv`, `mtl5.cholesky`, `mtl5.solve`.

**From C++:** [`phase07_sparse_formats/structured_views.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase07_sparse_formats/structured_views.cpp), [`phase10_matrix_views/triangular_views.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase10_matrix_views/triangular_views.cpp), [`phase10_matrix_views/permutation_block_diagonal.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase10_matrix_views/permutation_block_diagonal.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

import mtl5

V = mtl5.view

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

## A view in Python is a new matrix

In C++, a view is non-owning and costs nothing: `upper(A)` reads `A`'s storage
and hides the lower triangle. A Python caller gets a new matrix that holds the
view's values. The structure is the same, but the values are copied. So a later
change to `A` doesn't reach a view taken earlier. Take the view once the data is
final.

In [ ]:
a = np.arange(1.0, 17.0).reshape(4, 4)
A = mtl5.matrix(a)
U = V.upper(A)

a[0, 1] = 99.0  # change A after taking the view
print("A[0, 1] =", A.to_numpy()[0, 1], "  upper(A)[0, 1] =", U.to_numpy()[0, 1])
assert U.to_numpy()[0, 1] == 2.0
assert not np.shares_memory(U.to_numpy(), a)
a[0, 1] = 2.0

## Four triangles

`upper` and `lower` keep the diagonal, and `strict_upper` and `strict_lower`
drop it. So a matrix splits two ways, and both sums are exact: no arithmetic
other than adding zeros happens.

In [ ]:
L, SU = V.lower(A).to_numpy(), V.strict_upper(A).to_numpy()
Up, SL = V.upper(A).to_numpy(), V.strict_lower(A).to_numpy()

assert np.array_equal(L + SU, a)  # A = lower(A) + strict_upper(A)
assert np.array_equal(Up + SL, a)  # A = upper(A) + strict_lower(A)
assert np.array_equal(Up, np.triu(a)) and np.array_equal(SL, np.tril(a, -1))
print("A = lower + strict_upper = upper + strict_lower, exactly")

## The structures, side by side

The tutorial's 6 × 6 matrix has a diagonal 10, 11, … and off-diagonal entries
1 / (1 + |i − j|). Each panel below is one view of it, with the entries the
view sets to zero left blank.

In [ ]:
n = 6
i, j = np.indices((n, n))
full = 1.0 / (1.0 + np.abs(i - j))
np.fill_diagonal(full, 10.0 + np.arange(n))
F = mtl5.matrix(full)

panels = {
    "A": full,
    "upper(A)": V.upper(F).to_numpy(),
    "strict_lower(A)": V.strict_lower(F).to_numpy(),
    "banded(A, 1, 1)": V.banded(F, 1, 1).to_numpy(),
    "banded(A, 0, 2)": V.banded(F, 0, 2).to_numpy(),
    "map(A, rows 5..0)": V.map(F, list(range(n - 1, -1, -1)), list(range(n))).to_numpy(),
}

fig, axes = plt.subplots(2, 3, figsize=(7.5, 5.2))
for ax, (title, M) in zip(axes.flat, panels.items()):
    ax.imshow(np.ma.masked_equal(M, 0.0), cmap="Blues", norm=LogNorm(vmin=0.05, vmax=16.0))
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
fig.suptitle("Six views of one 6 × 6 matrix (blank = zero, color on a log scale)")
plt.tight_layout()
plt.show()

band = panels["banded(A, 1, 1)"]
assert np.array_equal(band, np.where(np.abs(i - j) <= 1, full, 0.0))
assert np.array_equal(V.banded(F, 0, 0).to_numpy(), np.diag(np.diag(full)))

## Store one triangle, solve with the whole matrix

A symmetric matrix needs only one triangle stored. `hermitian` reads the
upper triangle and mirrors it below; for complex matrices it conjugates the
mirror. The result is symmetric positive definite here, so a Cholesky
factorization solves it.

In [ ]:
upper_only = np.triu(full)  # the lower triangle is never stored
upper_only[np.diag_indices(n)] = 4.0 + np.arange(n)
H = V.hermitian(mtl5.matrix(upper_only))
h = H.to_numpy()

assert np.array_equal(h, h.T)
assert np.array_equal(np.triu(h), upper_only)
assert mtl5.is_spd(H)

b = 1.0 + np.arange(n)
x = mtl5.cholesky(H).solve(b).to_numpy()
residual = np.linalg.norm(h @ x - b)
print(f"Cholesky on the Hermitian view: residual {residual:.1e}")
assert residual < 1e-13

## Triangular solves

A lower-triangular system solves by forward substitution, one unknown per row.
`mtl5.trsv` does that in place on the right-hand side, and `upper=False` says
which triangle to use.

In [ ]:
L3 = V.lower(mtl5.matrix(np.array([[2.0, 0.0, 0.0], [3.0, 4.0, 0.0], [1.0, 5.0, 6.0]])))
rhs = np.array([2.0, 11.0, 29.0])
mtl5.trsv(L3, mtl5.vector(rhs), upper=False)  # rhs now holds the solution
print("x =", rhs)
assert np.array_equal(rhs, [1.0, 2.0, 3.0])

## Permutations are index vectors

A permutation matrix P with P[i, perm[i]] = 1 is never stored as a matrix. It
is the vector `perm`, and applying it costs O(n): (Px)[i] = x[perm[i]]. `map`
applies the same index vector to a matrix's rows, which is the product PA. The
inverse permutation is `argsort(perm)`, and it undoes P exactly.

In [ ]:
perm = [2, 0, 3, 1]
x = np.array([10.0, 20.0, 30.0, 40.0])
Px = x[perm]
inverse = np.argsort(perm)
print("P x =", Px, "  P^-1 P x =", Px[inverse])
assert np.array_equal(Px[inverse], x)

PA = V.map(A, perm, list(range(4))).to_numpy()  # rows of A in the order perm
P = np.eye(4)[perm]
assert np.array_equal(PA, P @ a)
assert np.array_equal(V.map(A, [1, 3], [0, 2]).to_numpy(), a[np.ix_([1, 3], [0, 2])])

## Block-diagonal systems decouple

A block-diagonal matrix is independent subsystems written as one matrix. The
tutorial's has a 2 × 2 rotation and a 3 × 3 tridiagonal block. Solving each
block alone gives the same answer as solving the whole matrix, at the cost of
the blocks: 2³ + 3³ = 35 instead of 5³ = 125 for an elimination. That is the
idea behind a block-Jacobi preconditioner, `mtl5.sparse.block_diagonal`.

In [ ]:
B1 = np.array([[0.8, -0.6], [0.6, 0.8]])
B2 = np.array([[2.0, -1.0, 0.0], [-1.0, 2.0, -1.0], [0.0, -1.0, 2.0]])
BD = np.zeros((5, 5))
BD[:2, :2], BD[2:, 2:] = B1, B2
rhs = np.array([0.8, 0.6, 1.0, 0.0, 1.0])

by_blocks = np.concatenate([mtl5.solve(B1, rhs[:2]), mtl5.solve(B2, rhs[2:])])
whole = mtl5.solve(BD, rhs)
print("by blocks:", by_blocks, "\nwhole:    ", whole)
assert np.allclose(by_blocks, whole, rtol=0, atol=1e-15)
assert np.allclose(by_blocks, [1.0, 0.0, 1.0, 1.0, 1.0], rtol=0, atol=1e-15)

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `upper(A)`, `lower(A)`, `strict_upper(A)`, `strict_lower(A)` | `mtl5.view.upper(A)`, and so on: a new matrix with the view's values |
| `triu(A, k)`, `tril(A, k)` | `np.triu`, `np.tril` on the buffer, or `mtl5.view.banded` |
| `banded(A, lower, upper)` | `mtl5.view.banded(A, lower, upper)` |
| `hermitian(A)` | `mtl5.view.hermitian(A)` |
| `mapped(A, rows, cols)` | `mtl5.view.map(A, rows, cols)` |
| `lower_trisolve(L, x, b)` | `mtl5.trsv(L, x, upper=False)`, in place |
| `permutation_matrix<double> P(perm)`, `P * x` | `x[perm]`; the inverse is `np.argsort(perm)` |
| `block_diagonal2D<double> BD({B1, B2})` | blocks solved one by one; `mtl5.sparse.block_diagonal` as a preconditioner |

Two C++ parts have no Python counterpart. A view's zero cost: a Python view is
a new matrix. And the packed L\U of `lu_factor`: `mtl5.lu` returns a factor
object that solves, and doesn't expose L and U.